In [1]:
import os
import sys
import json
from pathlib import Path
from typing import Any, Dict, List, Optional
import logging

import numpy as np
import pandas as pd
from sqlalchemy.orm import Session

# konfiguracja logowania w notebooku
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("dataset-notebook")

# ustawienie ścieżki do backendu, żeby móc importować moduły app.*
PROJECT_ROOT = Path("..").resolve()
BACKEND_PATH = PROJECT_ROOT / "backend"
if str(BACKEND_PATH) not in sys.path:
    sys.path.insert(0, str(BACKEND_PATH))

from app.database import SessionLocal
from app.models.user import User
from app.models.symptom_profile import SymptomProfile
from app.services.embedding_service import generate_embedding, generate_embeddings_batch
from app.services.matching_service import find_matching_group, add_user_to_group
from app.services.group_characteristics import update_group_characteristics
from app.routers.auth import _hash_password

DATA_PATH = PROJECT_ROOT / "data" / "test_data.json"
DRY_RUN = False

/home/mario/miniconda3/envs/zebrapoint/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from contextlib import contextmanager
from collections import Counter


@contextmanager
def get_db() -> Session:
    """Kontekst menedżer na sesję DB (tak jak w backendzie, ale dla notebooka)."""
    db = SessionLocal()
    try:
        yield db
        db.commit()
    except Exception:
        db.rollback()
        raise
    finally:
        db.close()


def load_dataset(path: Path) -> List[Dict[str, Any]]:
    """Wczytuje testowy zbiór danych z JSON-a."""
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError("Plik JSON powinien zawierać listę obiektów")

    records: List[Dict[str, Any]] = []
    for idx, raw in enumerate(data, start=1):
        email = raw.get("email")
        password = raw.get("haslo")
        description = raw.get("opis")
        display_name = raw.get("display_name")

        if not email or not password or not description:
            raise ValueError(f"Rekord {idx} ma brakujące pola (email/haslo/opis)")

        # display_name jest wymagany zgodnie z planem; jeśli go nie ma,
        # można doraźnie użyć prefiksu z emaila.
        if not display_name:
            display_name = email.split("@")[0]

        description = str(description).strip()
        if len(description) < 100:
            raise ValueError(
                f"Opis w rekordzie {idx} jest za krótki ({len(description)} znaków). "
                "Potrzebne >= 100 znaków."
            )

        rec = {
            "email": str(email).strip(),
            "password": str(password),
            "display_name": str(display_name).strip(),
            "description": description,
        }
        if raw.get("age_range") is not None:
            rec["age_range"] = str(raw["age_range"]).strip()
        records.append(rec)

    logger.info("Wczytano %d rekordów z %s", len(records), path)
    return records


def create_user_from_record(db: Session, record: Dict[str, Any]) -> User:
    """Tworzy użytkownika na podstawie rekordu z JSON-a albo zwraca istniejącego."""
    email = record["email"]
    display_name = record["display_name"]
    password = record["password"]

    user = db.query(User).filter(User.email == email).first()
    if user:
        logger.info("Użytkownik %s już istnieje — pomijam tworzenie", email)
        return user

    user = User(
        email=email,
        password_hash=_hash_password(password),
        display_name=display_name,
    )
    if record.get("age_range") is not None:
        user.age_range = record.get("age_range")
    db.add(user)
    db.flush()
    logger.info("Utworzono użytkownika %s", email)
    return user


def create_or_update_symptom_profile(
    db: Session,
    user: User,
    description: str,
) -> SymptomProfile:
    """Tworzy lub aktualizuje profil objawów i dopasowuje grupę."""
    profile = (
        db.query(SymptomProfile)
        .filter(SymptomProfile.user_id == user.id)
        .first()
    )

    embedding = generate_embedding(description)
    match = find_matching_group(db, embedding, user.id)
    group_id = match["group_id"]

    if profile is None:
        profile = SymptomProfile(
            user_id=user.id,
            description=description,
            embedding=embedding,
            group_id=group_id,
            match_score=match["score"],
        )
        db.add(profile)
        logger.info("Utworzono profil objawów dla %s", user.email)
    else:
        profile.description = description
        profile.embedding = embedding
        profile.group_id = group_id
        profile.match_score = match["score"]
        logger.info("Zaktualizowano profil objawów dla %s", user.email)

    add_user_to_group(db, user.id, group_id)

    db.flush()
    return profile

In [3]:
# Test połączenia z bazą Supabase

with get_db() as db:
    try:
        user_count = db.query(User).count()
        print(f"Połączenie OK. Liczba użytkowników w bazie: {user_count}")
    except Exception as exc:
        print("Błąd połączenia z bazą:", exc)
        raise

/tmp/ipykernel_45274/2424083691.py:5: SAWarning: relationship 'Comment.reactions' will copy column comments.id to column reactions.target_id, which conflicts with relationship(s): 'Post.reactions' (copies posts.id to reactions.target_id). If this is not the intention, consider if these relationships should be linked with back_populates, or if viewonly=True should be applied to one or more if they are read-only. For the less common case that foreign key constraints are partially overlapping, the orm.foreign() annotation can be used to isolate the columns that should be written towards.   To silence this warning, add the parameter 'overlaps="reactions"' to the 'Comment.reactions' relationship. (Background on this warning at: https://sqlalche.me/e/20/qzyx) (This warning originated from the `configure_mappers()` process, which was invoked automatically in response to a user-initiated operation.)
  user_count = db.query(User).count()


2026-03-14 20:47:06,500 INFO sqlalchemy.engine.Engine select pg_catalog.version()


INFO:sqlalchemy.engine.Engine:select pg_catalog.version()


2026-03-14 20:47:06,501 INFO sqlalchemy.engine.Engine [raw sql] {}


INFO:sqlalchemy.engine.Engine:[raw sql] {}


...

INFO:sqlalchemy.engine.Engine:SELECT count(*) AS count_1 
FROM (SELECT users.id AS users_id, users.email AS users_email, users.password_hash AS users_password_hash, users.display_name AS users_display_name, users.avatar_url AS users_avatar_url, users.role AS users_role, users.age_range AS users_age_range, users.is_active AS users_is_active, users.created_at AS users_created_at, users.updated_at AS users_updated_at 
FROM users) AS anon_1


2026-03-14 20:47:06,697 INFO sqlalchemy.engine.Engine [generated in 0.00096s] {}


INFO:sqlalchemy.engine.Engine:[generated in 0.00096s] {}


Połączenie OK. Liczba użytkowników w bazie: 0
2026-03-14 20:47:06,769 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


In [4]:
# Wczytanie danych z JSON i podgląd struktury

records = load_dataset(DATA_PATH)
print(f"Liczba rekordów w zbiorze: {len(records)}")

df_preview = pd.DataFrame(records)
df_preview.head()

INFO:dataset-notebook:Wczytano 40 rekordów z /home/mario/workspace/zebrapoint/data/test_data.json


Liczba rekordów w zbiorze: 40


,email,password,display_name,description
0,test0001@zebra.pl,test1234,Mama 1,Syn od urodzenia ma bardzo obniżone napięcie m...
1,test0002@zebra.pl,test1234,Mama 2,Córka ma zdiagnozowaną rzadką chorobę metaboli...
2,test0003@zebra.pl,test1234,Mama 3,"Nasze dziecko urodziło się z wadą serca, która..."
3,test0004@zebra.pl,test1234,Mama 4,Nasze dziecko urodziło się z bardzo niską masą...
4,test0005@zebra.pl,test1234,Mama 5,Od urodzenia zauważyliśmy u córki bardzo jasną...


In [5]:
# Utworzenie użytkowników w tabeli `users` na podstawie rekordów

created = 0
existing = 0

with get_db() as db:
    for rec in records:
        user_before = db.query(User).filter(User.email == rec["email"]).first()
        user = create_user_from_record(db, rec)
        if user_before is None:
            created += 1
        else:
            existing += 1

print(f"Nowo utworzonych użytkowników: {created}")
print(f"Użytkownicy już istniejący:   {existing}")

2026-03-14 20:47:27,190 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-03-14 20:47:27,194 INFO sqlalchemy.engine.Engine SELECT users.id AS users_id, users.email AS users_email, users.password_hash AS users_password_hash, users.display_name AS users_display_name, users.avatar_url AS users_avatar_url, users.role AS users_role, users.age_range AS users_age_range, users.is_active AS users_is_active, users.created_at AS users_created_at, users.updated_at AS users_updated_at 
FROM users 
WHERE users.email = %(email_1)s 
 LIMIT %(param_1)s


INFO:sqlalchemy.engine.Engine:SELECT users.id AS users_id, users.email AS users_email, users.password_hash AS users_password_hash, users.display_name AS users_display_name, users.avatar_url AS users_avatar_url, users.role AS users_role, users.age_range AS users_age_range, users.is_active AS users_is_active, users.created_at AS users_created_at, users.updated_at AS users_updated_at 
FROM users 
WHERE users.email = %(email_1)s 
 LIMIT %(param_1)s


2026-03-14 20:47:27,195 INFO sqlalchemy.engine.Engine [generated in 0.00140s] {'email_1': 'test0001@zebra.pl', 'param_1': 1}


...

2026-03-14 20:47:41,279 INFO sqlalchemy.engine.Engine [cached since 13.72s ago] {'id': UUID('fc285e54-cb25-423e-bce2-d2216733d294'), 'email': 'test0040@zebra.pl', 'password_hash': '$2b$12$LDXrCCSyIQMG7334seClD.yLNwoS9Aw5vToToMJIVayxa8owYWzC2', 'display_name': 'Mama 40', 'avatar_url': None, 'role': 'user', 'age_range': None, 'is_active': True}


INFO:sqlalchemy.engine.Engine:[cached since 13.72s ago] {'id': UUID('fc285e54-cb25-423e-bce2-d2216733d294'), 'email': 'test0040@zebra.pl', 'password_hash': '$2b$12$LDXrCCSyIQMG7334seClD.yLNwoS9Aw5vToToMJIVayxa8owYWzC2', 'display_name': 'Mama 40', 'avatar_url': None, 'role': 'user', 'age_range': None, 'is_active': True}
INFO:dataset-notebook:Utworzono użytkownika test0040@zebra.pl


2026-03-14 20:47:41,311 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


Nowo utworzonych użytkowników: 40
Użytkownicy już istniejący:   0


In [6]:
# Utworzenie / aktualizacja profili objawów i dopasowanie do grup

profiles_created = 0
profiles_updated = 0

with get_db() as db:
    for rec in records:
        email = rec["email"]
        user = db.query(User).filter(User.email == email).first()
        if user is None:
            logger.warning("Brak użytkownika %s — pomijam profil objawów", email)
            continue

        existing_profile = (
            db.query(SymptomProfile)
            .filter(SymptomProfile.user_id == user.id)
            .first()
        )

        profile = create_or_update_symptom_profile(db, user, rec["description"])

        if existing_profile is None:
            profiles_created += 1
        else:
            profiles_updated += 1

    # Uzupełnienie charakterystyk grup (keywords, symptom_category, avg_match_score),
    # gdy Celery nie działa — update_group_characteristics robi commit wewnętrznie.
    group_ids = {row[0] for row in db.query(SymptomProfile.group_id).all() if row[0] is not None}
    for gid in group_ids:
        update_group_characteristics(db, str(gid))

print(f"Nowo utworzonych profili objawów: {profiles_created}")
print(f"Zaktualizowanych profili objawów: {profiles_updated}")

2026-03-14 20:48:25,648 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-03-14 20:48:25,650 INFO sqlalchemy.engine.Engine SELECT users.id AS users_id, users.email AS users_email, users.password_hash AS users_password_hash, users.display_name AS users_display_name, users.avatar_url AS users_avatar_url, users.role AS users_role, users.age_range AS users_age_range, users.is_active AS users_is_active, users.created_at AS users_created_at, users.updated_at AS users_updated_at 
FROM users 
WHERE users.email = %(email_1)s 
 LIMIT %(param_1)s


INFO:sqlalchemy.engine.Engine:SELECT users.id AS users_id, users.email AS users_email, users.password_hash AS users_password_hash, users.display_name AS users_display_name, users.avatar_url AS users_avatar_url, users.role AS users_role, users.age_range AS users_age_range, users.is_active AS users_is_active, users.created_at AS users_created_at, users.updated_at AS users_updated_at 
FROM users 
WHERE users.email = %(email_1)s 
 LIMIT %(param_1)s


2026-03-14 20:48:25,650 INFO sqlalchemy.engine.Engine [cached since 58.46s ago] {'email_1': 'test0001@zebra.pl', 'param_1': 1}


...

2026-03-14 20:48:53,542 INFO sqlalchemy.engine.Engine [cached since 6.316s ago] {'pk_1': UUID('6c079d41-d7ef-4c7f-a66d-f116425206d7')}


INFO:sqlalchemy.engine.Engine:[cached since 6.316s ago] {'pk_1': UUID('6c079d41-d7ef-4c7f-a66d-f116425206d7')}
INFO:app.services.group_characteristics:Charakterystyki grupy 6c079d41-d7ef-4c7f-a66d-f116425206d7 zaktualizowane: keywords=['napady', 'niemowlęctwa', 'drgawek', 'rozwój', 'ruchowy'], category=Neurologiczne, age_range=None, avg_score=0.0


2026-03-14 20:48:53,604 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


Nowo utworzonych profili objawów: 40
Zaktualizowanych profili objawów: 0


In [7]:
# Diagnostyka: liczba profili i rozkład po grupach

with get_db() as db:
    total_profiles = db.query(SymptomProfile).count()
    print(f"Liczba profili objawów w bazie: {total_profiles}")

    rows = (
        db.query(SymptomProfile.group_id)
        .all()
    )


group_ids = [row[0] for row in rows]
counts = Counter(group_ids)

print("\nRozkład liczby profili w grupach:")
for group_id, cnt in counts.items():
    print(f"  grupa={group_id}: {cnt} profili")

2026-03-14 20:49:42,708 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-03-14 20:49:42,711 INFO sqlalchemy.engine.Engine SELECT count(*) AS count_1 
FROM (SELECT symptom_profiles.id AS symptom_profiles_id, symptom_profiles.user_id AS symptom_profiles_user_id, symptom_profiles.description AS symptom_profiles_description, symptom_profiles.embedding AS symptom_profiles_embedding, symptom_profiles.group_id AS symptom_profiles_group_id, symptom_profiles.match_score AS symptom_profiles_match_score, symptom_profiles.created_at AS symptom_profiles_created_at, symptom_profiles.updated_at AS symptom_profiles_updated_at 
FROM symptom_profiles) AS anon_1


INFO:sqlalchemy.engine.Engine:SELECT count(*) AS count_1 
FROM (SELECT symptom_profiles.id AS symptom_profiles_id, symptom_profiles.user_id AS symptom_profiles_user_id, symptom_profiles.description AS symptom_profiles_description, symptom_profiles.embedding AS symptom_profiles_embedding, symptom_profiles.group_id AS symptom_profiles_group_id, symptom_profiles.match_score AS symptom_profiles_match_score, symptom_profiles.created_at AS symptom_profiles_created_at, symptom_profiles.updated_at AS symptom_profiles_updated_at 
FROM symptom_profiles) AS anon_1


2026-03-14 20:49:42,713 INFO sqlalchemy.engine.Engine [generated in 0.00230s] {}


...

2026-03-14 20:49:42,774 INFO sqlalchemy.engine.Engine [cached since 55.78s ago] {}


INFO:sqlalchemy.engine.Engine:[cached since 55.78s ago] {}


2026-03-14 20:49:42,805 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT



Rozkład liczby profili w grupach:
  grupa=a2644a80-31b4-430d-9a1d-7d1cbc922e9b: 9 profili
  grupa=ee9f537e-dc5a-4927-ac9c-7fb86211aa09: 1 profili
  grupa=e816d3a6-a075-45f7-92c4-3d1f38818ea9: 1 profili
  grupa=49357027-f555-473a-8c72-cafd78dc5fc2: 3 profili
  grupa=3c8705ba-ea53-4263-b938-42b64aee1feb: 1 profili
  grupa=0946e26a-2416-427b-80d3-cb4e30902b5b: 3 profili
  grupa=bb90910c-36eb-4467-ba7d-1b5b5e4ed1ee: 1 profili
  grupa=e9a71a4c-b2de-4a87-ac05-4d527d6745ab: 1 profili
  grupa=a3bc3fbb-f3a1-4c34-906e-a6676e30d21c: 2 profili
  grupa=381f180d-adc3-4a20-a80a-76d8ef0793ff: 1 profili
  grupa=e1fc2c14-04c6-4cd7-b932-7805776a8ba9: 1 profili
  grupa=ccfcab6e-ad44-4c44-a3f4-f89c9067e6ef: 2 profili
  grupa=ad488d0b-6569-40df-ae7a-3ee3758b2236: 1 profili
  grupa=70b3efbb-3c9b-44db-bf24-f453515ab3dd: 2 profili
  grupa=965b3861-f29a-43cd-8e70-6474097fafa6: 1 profili
  grupa=ce577c16-4dd0-445e-8e55-96b461abf41c: 1 profili
  grupa=b239d74f-86fd-4a12-b3d4-ade068475954: 1 profili
  grupa=a14e3

In [8]:
# Eksperyment ML: prosta klasteryzacja KMeans na embeddingach

from sklearn.cluster import KMeans

# Weźmy embeddingi dla opisów z JSON-a (bez czytania z bazy)
texts = [rec["description"] for rec in records]
embeddings = generate_embeddings_batch(texts)

X = np.array(embeddings)
print("Macierz embeddingów:", X.shape)

# Przy niewielkiej liczbie rekordów spróbujmy np. 3 klastry
n_clusters = 3
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init="auto")
cluster_labels = kmeans.fit_predict(X)

# Podgląd wyników w DataFrame
ml_df = pd.DataFrame(
    {
        "email": [rec["email"] for rec in records],
        "cluster": cluster_labels,
    }
)

display(ml_df.head())
print("\nLiczność klastrów:")
print(ml_df["cluster"].value_counts())

Batches: 100%|██████████| 2/2 [00:01<00:00,  1.74it/s]

Macierz embeddingów: (40, 384)


,email,cluster
0,test0001@zebra.pl,1
1,test0002@zebra.pl,2
2,test0003@zebra.pl,2
3,test0004@zebra.pl,1
4,test0005@zebra.pl,0



Liczność klastrów:
cluster
1    17
0    13
2    10
Name: count, dtype: int64


In [9]:
def run_full_seed(dry_run: bool = False) -> None:
    """Wysokopoziomowa funkcja do ponownego uruchamiania całego seeda.

    Kroki:
    - wczytanie JSON,
    - stworzenie użytkowników,
    - stworzenie / aktualizacja profili objawów i dopasowanie grup.
    """
    global DRY_RUN, records
    DRY_RUN = dry_run

    # wczytaj dane
    records = load_dataset(DATA_PATH)
    logger.info("Start seeda. Rekordów: %d (DRY_RUN=%s)", len(records), DRY_RUN)

    created = 0
    existing = 0
    profiles_created = 0
    profiles_updated = 0

    with get_db() as db:
        for rec in records:
            # użytkownik
            user_before = db.query(User).filter(User.email == rec["email"]).first()
            user = create_user_from_record(db, rec)
            if user_before is None:
                created += 1
            else:
                existing += 1

            # profil objawów
            existing_profile = (
                db.query(SymptomProfile)
                .filter(SymptomProfile.user_id == user.id)
                .first()
            )
            profile = create_or_update_symptom_profile(db, user, rec["description"])
            if existing_profile is None:
                profiles_created += 1
            else:
                profiles_updated += 1

        if not DRY_RUN:
            # Uzupełnienie charakterystyk grup (gdy Celery nie działa).
            group_ids = {row[0] for row in db.query(SymptomProfile.group_id).all() if row[0] is not None}
            for gid in group_ids:
                update_group_characteristics(db, str(gid))

        if DRY_RUN:
            logger.info("DRY_RUN=True — wykonuję rollback zmian")
            db.rollback()
        else:
            logger.info("Zatwierdzam zmiany w bazie")

    print("Seed zakończony:\n")
    print(f"  Nowo utworzonych użytkowników: {created}")
    print(f"  Użytkownicy już istniejący:   {existing}")
    print(f"  Nowo utworzonych profili:     {profiles_created}")
    print(f"  Zaktualizowanych profili:     {profiles_updated}")


# Przykład użycia:
# run_full_seed(dry_run=True)  # test bez zapisywania do bazy
# run_full_seed(dry_run=False) # realny seed do bazy